In [1]:
#CRIAÇÃO DA TABELA GOLD FACT SALES
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA FATO DE VENDAS
# ============================================================
#
# Objetivo:
# Criar a tabela fato de vendas (Fact Sales) a partir dos
# dados transacionais da camada Silver.
#
# Esta tabela armazenará os eventos de venda e será a principal
# fonte para análises comerciais, financeiras e operacionais.
#
# Transformações realizadas:
# - Renomeação das colunas para nomenclatura de negócio
# - Organização das chaves de relacionamento com dimensões
# - Disponibilização das métricas de vendas
#
# Esta tabela será utilizada para análises de:
#
# - Receita
# - Volume vendido
# - Ticket médio
# - Performance por produto
# - Performance por localização
# - Evolução temporal das vendas
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE gold__fact_sales__lite AS

SELECT

    -- Identificador único da transação de venda
    transaction_id AS SalesTransactionID,

    -- Data da venda
    transaction_date AS SalesDate,

    -- Chave de relacionamento com a dimensão de produtos
    sku AS ProductSKU,

    -- Chave de relacionamento com a dimensão de localidades
    location_id AS LocationID,

    -- Quantidade vendida
    qty AS Quantity,

    -- Preço unitário da venda
    unit_price AS UnitPrice

FROM silver__sales_transactions__lite
""")


# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "gold__fact_sales__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela gold__fact_sales__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela gold__fact_sales__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:41:55] ✅ Tabela gold__fact_sales__lite criada com sucesso!


In [2]:
#CRIAÇÃO DA TABELA GOLD FACT INVENTORY MOVEMENTS
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))

# ============================================================
# CRIAÇÃO DA TABELA FATO DE MOVIMENTAÇÕES DE ESTOQUE
# ============================================================
#
# Objetivo:
# Criar a tabela fato de movimentações de estoque
# (Fact Inventory Movements) a partir dos dados tratados
# na camada Silver.
#
# Esta tabela registra todos os eventos de movimentação
# ocorridos ao longo da cadeia logística, permitindo
# rastrear entradas, saídas e transferências de produtos.
#
# Transformações realizadas:
# - Renomeação das colunas para nomenclatura de negócio
# - Organização das chaves de relacionamento
# - Disponibilização das métricas de movimentação
#
# Esta tabela será utilizada para análises de:
#
# - Fluxo de estoque
# - Transferências entre localidades
# - Entradas e saídas de produtos
# - Movimentações por período
# - Auditoria logística
# - Rastreabilidade operacional
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE gold__fact_inventory_movements__lite AS

SELECT

    -- Identificador único da movimentação de estoque
    movement_id AS InventoryMovementID,

    -- Data da movimentação
    movement_date AS MovementDate,

    -- Tipo da movimentação
    -- Ex.: RECEIPT, SHIPMENT, TRANSFER, ADJUSTMENT
    movement_type AS MovementType,

    -- Chave de relacionamento com a dimensão de produtos
    sku AS ProductSKU,

    -- Localidade de origem da movimentação
    from_location_id AS FromLocationID,

    -- Localidade de destino da movimentação
    to_location_id AS ToLocationID,

    -- Quantidade movimentada
    qty AS Quantity

FROM silver__inventory_movements__lite;
""")


# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "gold__fact_inventory_movements__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela gold__fact_inventory_movements__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela gold__fact_inventory_movements__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:41:55] ✅ Tabela gold__fact_inventory_movements__lite criada com sucesso!


In [3]:
#CRIAÇÃO DA TABELA GOLD FACT INVENTORY SNAPSHOTS
from datetime import datetime
from pathlib import Path
import duckdb

# Procura o banco automaticamente : o notebook continuará encontrando o banco automaticamente dentro do projeto, 
# mesmo que ele seja movido para outra pasta ou máquina, desde que a estrutura do projeto seja mantida.
base_dir = Path.cwd().parent

db_path = next(base_dir.rglob("supply_chain_analytics.duckdb"))

conn = duckdb.connect(str(db_path))


# ============================================================
# CRIAÇÃO DA TABELA FATO DE POSIÇÃO DE ESTOQUE
# ============================================================
#
# Objetivo:
# Criar a tabela fato de snapshots de estoque
# (Fact Inventory Snapshots) a partir dos dados tratados
# na camada Silver.
#
# Esta tabela representa a posição do estoque em uma
# determinada data, permitindo acompanhar a evolução dos
# níveis de inventário ao longo do tempo.
#
# Diferentemente das movimentações de estoque, que registram
# eventos, esta tabela representa o estado do estoque em um
# momento específico.
#
# Transformações realizadas:
# - Renomeação das colunas para nomenclatura de negócio
# - Organização das chaves de relacionamento
# - Disponibilização da métrica de estoque disponível
#
# Esta tabela será utilizada para análises de:
#
# - Nível de estoque
# - Cobertura de estoque
# - Evolução do inventário
# - Risco de ruptura
# - Excesso de estoque
# - Valorização do inventário
#
# ============================================================

conn.execute("""
CREATE OR REPLACE TABLE gold__fact_inventory_snapshots__lite AS

SELECT

    -- Data de referência do snapshot
    snapshot_date AS SnapshotDate,

    -- Chave de relacionamento com a dimensão de localidades
    location_id AS LocationID,

    -- Chave de relacionamento com a dimensão de produtos
    sku AS ProductSKU,

    -- Quantidade disponível em estoque na data do snapshot
    on_hand_qty AS OnHandQuantity

FROM silver__inventory_snapshots__lite;
""")


# Verifica se a tabela foi criada com sucesso
tables = conn.sql("SHOW TABLES").df()

if "gold__fact_inventory_snapshots__lite" in tables["name"].values:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"✅ Tabela gold__fact_inventory_snapshots__lite criada com sucesso!"
    )
else:
    print(
        f"[{datetime.now():%d/%m/%Y %H:%M:%S}] "
        f"❌ Tabela gold__fact_inventory_snapshots__lite não foi criada!"
    )
conn.close()

[03/06/2026 23:41:56] ✅ Tabela gold__fact_inventory_snapshots__lite criada com sucesso!
